# Manuscript Figures 13–17

This notebook reproduces the five finite-element **plotting figures** from the supplied processed simulation data. The numerical simulations are **not** rerun. Each figure has its own code cell to simplify later edits.

**Requirements:** Python 3, NumPy, pandas, Matplotlib, and Jupyter. Open this notebook from the directory containing the `data/` folder. Execute the setup cells first, then each figure cell. Outputs are shown inline and written as vector PDF and 300-dpi PNG files in `figures/`.

The filtered data, theory samples, graphing conventions, and plotted coordinates follow the updated plotting notebooks supplied October 22, 2026. 

## Setup and data paths

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Start Jupyter from the directory containing this notebook and data/.
DATA_DIR = Path("data")
FIG_DIR = Path("figures")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        "Cannot find data/. Launch Jupyter in the directory containing "
        "Figures13_17.ipynb and data/."
    )
FIG_DIR.mkdir(exist_ok=True)

PI = np.pi
print("Reading CSV files from:", DATA_DIR.resolve())
print("Writing figures to:", FIG_DIR.resolve())


In [ ]:
def set_figure_style(font_size=26):
    """Original plotting settings, with serif fallbacks for other computers."""
    mpl.rcdefaults()
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "STIX Two Text", "STIXGeneral"],
        "mathtext.fontset": "stix",
        "font.size": font_size,
        "axes.labelsize": 28,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
        "legend.fontsize": 22,
        "axes.linewidth": 1.2,
        "xtick.major.width": 1.2,
        "ytick.major.width": 1.2,
        "xtick.major.size": 6,
        "ytick.major.size": 6,
    })


def export_figure(fig, number):
    """Save the current Figure N using paths relative to this notebook."""
    pdf_path = FIG_DIR / f"Fig{number}.pdf"
    png_path = FIG_DIR / f"Fig{number}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    print(f"Saved {pdf_path} and {png_path}")


## Figure 13 — Circular inclusion: collapse versus unwinding

Classification of simulations (blue circles for separation, orange crosses for collapse) and the existing 180-point theoretical curve from Mathematica. Original selection criteria: max_delta at least 50 and min_delta at most −50. No smoothing of the theoretical line.

In [ ]:
set_figure_style(font_size=22)

phase_data = pd.read_csv(DATA_DIR / "critical_phi_calculated_defects_data.csv")
phase_data = phase_data.loc[
    (phase_data["max_delta"] >= 50) & (phase_data["min_delta"] <= -50)
].copy()
phase_data["dphi"] = phase_data["phi_max"] - phase_data["phi_min"]
phase_theory = pd.read_csv(DATA_DIR / "fig9_mathematica_data.csv")

rho = 1 / np.e
initial_r = np.sqrt(rho)
alpha_values = np.linspace(0.6, 1.0, 5) * PI
phi_values = np.linspace(0.02, 0.07, 11) * PI

fig, ax = plt.subplots(figsize=(10, 8))
marker_count = 0
for phi in phi_values:
    for alpha in alpha_values:
        subset = phase_data.loc[
            np.isclose(phase_data["alpha0"], alpha)
            & np.isclose(phase_data["phi_param"], phi)
            & np.isclose(phase_data["r_param"], initial_r)
        ]
        if subset.empty:
            raise ValueError(f"Figure 13: missing simulation data at alpha={alpha}, phi={phi}")
        if subset["dphi"].max() > PI:
            ax.scatter(alpha / PI, 2 * phi / PI, color="tab:blue")
        else:
            ax.scatter(alpha / PI, 2 * phi / PI, color="tab:orange", marker="x",linewidths=2)
        marker_count += 1

ax.plot(phase_theory["alphabypi"], phase_theory["dphibypi"],
        color="black", label="Theoretical prediction", linewidth=2.5)
ax.set_xlabel(r"$\alpha/\pi$")
ax.set_ylabel(r"$\Delta\phi/\pi$")
ax.set_xlim(0.55, 1.05)
ax.set_ylim(0.03, 0.15)
export_figure(fig, 13)
print(f"Plotted {marker_count} classification markers and {len(phase_theory)} theory points")
plt.show()


## Figure 14 — Circular inclusion: defect trajectories at alpha = 0.6 pi

Simulation trajectories for three annular widths (ell = 0.5, 1, 2) and the supplied theoretical optimal radial curves. Plot positive and negative defects separately as `+` and `.` respectively, exactly as in the source notebook. The numerical initial-angle parameter is `phi_param = 0.52` rad, approximately pi/6. Filtering threshold: 50.

In [ ]:
set_figure_style(font_size=22)

annulus_data_50 = pd.read_csv(DATA_DIR / "calculated_defects_data.csv")
annulus_data_50 = annulus_data_50.loc[
    (annulus_data_50["max_delta"] >= 50)
    & (annulus_data_50["min_delta"] <= -50)
].copy()
radial_theory = pd.read_csv(DATA_DIR / "fig7data_mathematica.csv")
rho_values = np.sort(annulus_data_50["rho"].unique())[::-1]
colors = ["tab:blue", "tab:orange", "tab:green"]
alpha = 0.6 * PI
phi = 0.52

fig, ax = plt.subplots(figsize=(10, 8))
for color, rho in zip(colors, rho_values):
    ell = np.log(1 / rho)
    initial_r = np.sqrt(rho)
    subset = annulus_data_50.loc[
        np.isclose(annulus_data_50["alpha0"], alpha)
        & np.isclose(annulus_data_50["r_param"], initial_r)
        & np.isclose(annulus_data_50["phi_param"], phi)
        & np.isclose(annulus_data_50["rho"], rho)
    ]
    if len(subset) <= 10:
        raise ValueError(f"Figure 14: missing or insufficient simulation data for ell={ell}")
    x_plus = 2 * subset["phi_max"] / PI
    x_minus = -2 * subset["phi_min"] / PI
    c = 1 / np.log(rho)
    ax.scatter(x_minus, 1 - c * np.log(subset["r_min"]),
           marker="o", color=color, s=25,
           label=rf"$\ell$ = {ell:.2f}")
    ax.scatter(x_plus, 1 - c * np.log(subset["r_max"]),
           marker="+", color=color, s=100, linewidths=1.5,
           label=rf"$\ell$ = {ell:.2f}")
    theory_subset = radial_theory.loc[np.isclose(radial_theory["l"], ell)]
    if theory_subset.empty:
        raise ValueError(f"Figure 14: missing radial theory for ell={ell}")
    ax.plot(theory_subset["phi"], theory_subset["f"], color=color,linewidth=2.5,
            label=rf"$\ell$ = {ell:.2f}")

ax.set_xlabel(r"$\Delta\phi/\pi,\ \pm 2\phi_\pm/\pi$")
ax.set_ylabel(r"$1-\log r_\pm /\log\rho$")
ax.set_ylim(0, 0.5)
# No legend in the supplied original.
export_figure(fig, 14)
plt.show()


## Figure 15 — Circular inclusion: defect trajectories at alpha = 0.8 pi

Same construction as Figure 14, but with alpha = 0.8 pi and the updated Annulus plotting notebook's threshold of 10. 

In [ ]:
set_figure_style(font_size=22)

annulus_data_10 = pd.read_csv(DATA_DIR / "calculated_defects_data.csv")
annulus_data_10 = annulus_data_10.loc[
    (annulus_data_10["max_delta"] >= 10)
    & (annulus_data_10["min_delta"] <= -10)
].copy()

rho_values = np.sort(annulus_data_10["rho"].unique())[::-1]
colors = ["tab:blue", "tab:orange", "tab:green"]

alpha = 0.8 * PI
phi = 0.52

fig, ax = plt.subplots(figsize=(10, 8))

for color, rho in zip(colors, rho_values):
    ell = np.log(1 / rho)
    initial_r = np.sqrt(rho)

    subset = annulus_data_10.loc[
        np.isclose(annulus_data_10["alpha0"], alpha)
        & np.isclose(annulus_data_10["r_param"], initial_r)
        & np.isclose(annulus_data_10["phi_param"], phi)
        & np.isclose(annulus_data_10["rho"], rho)
    ]

    if len(subset) <= 10:
        raise ValueError(f"Figure 15: missing or insufficient simulation data for ell={ell}")

    x_plus = 2 * subset["phi_max"] / PI
    x_minus = -2 * subset["phi_min"] / PI
    c = 1 / np.log(rho)

    ax.scatter(
        x_minus,
        1 - c * np.log(subset["r_min"]),
        marker="o",
        color=color, s=25
    )

    ax.scatter(
        x_plus,
        1 - c * np.log(subset["r_max"]),
        marker="+",
        color=color, s=100, linewidths=1.5
    )

    theory_subset = radial_theory.loc[np.isclose(radial_theory["l"], ell)]
    if theory_subset.empty:
        raise ValueError(f"Figure 15: missing radial theory for ell={ell}")

    ax.plot(
        theory_subset["phi"],
        theory_subset["f"],
        color=color,
        linewidth=2.5
    )

ax.set_xlabel(r"$\Delta\phi/\pi,\ \pm 2\phi_\pm/\pi$")
ax.set_ylabel(r"$1-\log r_\pm / \log \rho$")
ax.set_ylim(0, 0.5)

export_figure(fig, 15)
plt.show()

## Figure 16 — Circular inclusion: angular separation versus time

Six alpha values from 0.5 pi to pi, `rho = exp(-1)`, `phi_param = 0.26` rad (approximately pi/12), threshold 10. Time sorting and source colors are retained.

In [ ]:
set_figure_style(font_size=22)

rho = rho_values[1]  # The middle annulus, ell = 1, as in the source.
initial_r = np.sqrt(rho)
phi = 0.26
alpha_values = np.linspace(0.5, 1.0, 6) * PI
colors = ["tab:blue", "tab:orange", "teal", "xkcd:magenta", "tab:brown", "cyan"]
fig, ax = plt.subplots(figsize=(10, 8))

for color, alpha in zip(colors, alpha_values):
    subset = annulus_data_10.loc[
        np.isclose(annulus_data_10["alpha0"], alpha)
        & np.isclose(annulus_data_10["rho"], rho)
        & (annulus_data_10["phi_param"] == phi)
        & np.isclose(annulus_data_10["r_param"], initial_r)
    ].sort_values("t")
    if subset.empty:
        raise ValueError(f"Figure 16: missing trajectory for alpha/pi={alpha/PI}")
    ax.plot(subset["t"], (subset["phi_max"] - subset["phi_min"]) / PI,
            color=color, linewidth=2.5, label=rf"$\alpha_0$ = {alpha/PI:.2f}$\pi$")

ax.set_xlabel(r"$tA/\gamma$")
ax.set_ylabel(r"$\Delta\phi/\pi$")
# No legend in the supplied original.
export_figure(fig, 16)
plt.show()


## Figure 17 — Elliptical inclusion: angular separation versus time

This is for `u1 = 0.3`, `u2 = 2`.  Revised tracking threshold  `max_delta >= 100`, `min_delta <= -100`, and `t > 0` filters are used. 

In [ ]:
set_figure_style(font_size=26)

ellipse_data = pd.read_csv(DATA_DIR / "ellipse_defects_data.csv")
ellipse_data = ellipse_data.loc[
    (ellipse_data["max_delta"] >= 100)
    & (ellipse_data["min_delta"] <= -100)
    & (ellipse_data["t"] > 0)
].copy()

u1, u2 = 0.3, 2.0
initial_r = np.sqrt(np.cosh(u1) / np.cosh(u2))
phi = PI / 6
alpha_values = np.linspace(0.5, 1.0, 6) * PI
colors = ["tab:blue", "tab:orange", "teal", "xkcd:magenta", "tab:brown", "cyan"]
fig, ax = plt.subplots(figsize=(10, 8))

for color, alpha in zip(colors, alpha_values):
    subset = ellipse_data.loc[
        np.isclose(ellipse_data["alpha0"], alpha)
        & np.isclose(ellipse_data["phi_param"], phi)
        & np.isclose(ellipse_data["r_param"], initial_r)
        ].sort_values("t")
    if subset.empty:
        raise ValueError(f"Figure 17: missing trajectory for alpha/pi={alpha/PI}")
    # Source order is retained; no endpoint trimming or smoothing.
    ax.plot(subset["t"], (subset["phi_max"] - subset["phi_min"]) / PI,
            color=color, linewidth=2.5, label=rf"$\alpha_0$ = {alpha/PI:.2f}$\pi$")

ax.set_xlabel(r"$tA/\gamma$")
ax.set_ylabel(r"$\Delta\phi/\pi$")
export_figure(fig, 17)
plt.show()


## Provenance and interpretation

- **Figure 13:** `PythonCode/critical-phi/plotting.ipynb` and the phase-diagram/theory CSVs.
- **Figure 14:** `Simulation data and figures/plottingTRP.ipynb` and the annulus trajectory/radial-theory CSVs.
- **Figures 15–16:** `PythonCode/Annulus/annulus_plotting.ipynb` and the same two CSVs.
- **Figure 17:** `PythonCode/Ellipse/ellipse_plotting.ipynb` and the elliptical trajectory CSV.
